# Joins in SLayer

TL;DR: Use dotted join-path syntax everywhere — to reference measures/dimensions from joined models inside queries, AND inside the `sql` of dimension, measure, and model-filter definitions. The `__` spelling is only an emitted-SQL alias detail, never an input form; the legacy `__` split-alias input form is a hard error. 

Joins connect models so that dimensions and measures from one model are accessible when querying another. SLayer only supports **LEFT JOINs** — the kind most commonly used for data enrichment.

If you think of left joins as directed edges of a graph whose vertices are models, then the dimensions, measures, and filters in a model have access to columns from any model reachable in the join graph.

This notebook illustrates every aspect of joins with working code.

See also: [Models — Joins](../../concepts/models.md#joins) | [Queries — Cross-Model Measures](../../concepts/queries.md#cross-model-measures) | [Ingestion — Diamond Joins](../../concepts/ingestion.md#diamond-joins)

**Prerequisites:** `pip install motley-slayer`

In [1]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

engine, storage, models = ensure_jaffle_shop()

## Basic Joins

A join is defined by a **target model** and a list of **join pairs** — column pairs to join on. The `orders` model has two joins, auto-generated from its foreign keys to `customers` and `stores`:

In [2]:
orders_model = next(m for m in models if m.name == "orders")

print("orders model joins:")
for j in orders_model.joins:
    print(f"  -> {j.target_model}  ON {j.join_pairs}")

# Query using a joined dimension
result = engine.execute_sync(
    query={
        "source_model": "orders",
        "measures": ["count(*)", "sum(order_total)"],
        "dimensions": ["customers.name"],
        "order": [{"column": "order_total_sum", "direction": "desc"}],
        "limit": 5,
    }
)

print("\nTop 5 customers by revenue:")
for row in result.data:
    print(f"  {row['orders.customers.name']}: {row['orders._count']} orders, ${row['orders.order_total_sum']:,.2f}")

orders model joins:
  -> customers  ON [['customer_id', 'id']]
  -> stores  ON [['store_id', 'id']]

Top 5 customers by revenue:
  Rhonda Gomez: 153 orders, $14,223.69
  Bruce Cole: 149 orders, $13,938.62
  John Lowe: 832 orders, $13,785.26
  Christine Gonzales: 497 orders, $13,584.70
  Tanya Anderson: 142 orders, $13,314.28


## Referencing Joined Models in Queries (Dot Syntax)

SLayer uses **dot syntax** to reference dimensions and measures from joined models:

- 1-hop: `customers.name` (orders -> customers)
- Multi-hop: `orders.customers.name` (items -> orders -> customers)

The full path avoids ambiguity when there are multiple ways to reach a model.

In [3]:
# 1-hop: orders -> stores
result = engine.execute_sync(
    query={
        "source_model": "orders",
        "measures": ["count(*)"],
        "dimensions": ["stores.name"],
    }
)

print("1-hop (orders -> stores):")
for row in result.data:
    print(f"  {row['orders.stores.name']}: {row['orders._count']} orders")

1-hop (orders -> stores):
  New Orleans: 15686 orders
  San Francisco: 90833 orders
  Brooklyn: 254734 orders
  Chicago: 102403 orders
  Philadelphia: 195912 orders


In [4]:
# Multi-hop: items -> orders -> customers
result = engine.execute_sync(
    query={
        "source_model": "items",
        "measures": ["count(*)"],
        "dimensions": ["orders.customers.name"],
        "order": [{"column": "_count", "direction": "desc"}],
        "limit": 5,
    }
)

print("Multi-hop (items -> orders -> customers):")
for row in result.data:
    print(
        f"  {row['items.orders.customers.name']}: {row['items._count']} line items"
    )

Multi-hop (items -> orders -> customers):
  John Lowe: 1656 line items
  Rhonda Gomez: 1530 line items
  Bruce Cole: 1490 line items
  Christine Gonzales: 1465 line items
  Tanya Anderson: 1420 line items


## Referencing Joined Models in SQL Snippets

When defining dimensions, measures, and filters at the model level (in YAML or Python), their `sql` fields use the **same dotted join-path syntax** as queries. SLayer resolves the path through the join graph and emits the correct table aliases in the generated SQL.

| Context | Syntax | Example |
|---------|--------|---------|
| Query dimensions/measures | dots | `orders.customers.name` |
| Model SQL expressions | dots | `orders.customers.name` |

The generated SQL uses a `__`-delimited internal alias for each joined table (`LEFT JOIN customers AS orders__customers`), but that spelling is an emitted-SQL detail — you always write dotted paths. The legacy `__` split-alias input form (`orders__customers.name` meant as a join walk) is no longer accepted; it is a hard error.

In [5]:
# Model-level SQL uses the same dotted join-path syntax as queries.
# Auto-ingestion doesn't generate these — joined columns are resolved at query time.
#
# Here's a manual example:

from slayer.core.models import Column

manual_col = Column(
    name="customer_name",
    sql="orders.customers.name",  # dotted join path in model SQL
    type="string",
)
print(f"Model-level SQL (dotted path):   sql={manual_col.sql!r}")
print("Query-level (same dotted path):  orders.customers.name")

# SLayer resolves the dotted path to the correct __ internal alias in generated SQL:
result = engine.execute_sync(
    query={
        "source_model": "items",
        "measures": ["count(*)"],
        "dimensions": ["orders.customers.name"],
        "limit": 3,
    }
)

print(f"\nResult keys: {list(result.data[0].keys())}")
for row in result.data:
    print(f"  {row['items.orders.customers.name']}: {row['items._count']} items")


Model-level SQL (dotted path):   sql='orders.customers.name'
Query-level (same dotted path):  orders.customers.name



Result keys: ['items.orders.customers.name', 'items._count']
  Trevor Kim: 274 items
  Michele Cohen: 275 items
  Jasmine Reese: 407 items


## Auto-Ingesting Schemas

When auto-ingesting a database, SLayer introspects FK constraints and creates a `ModelJoin` for each direct FK on the source table. Multi-hop reachability (e.g. `items → orders → customers`) is resolved at query time by walking each intermediate model's own joins — it is not baked into the source model's join list.

In [6]:
import sqlalchemy as sa

from setup_jaffle import DB_PATH
from slayer.core.models import DatasourceConfig
from slayer.engine.ingestion import _build_fk_graph

ds = DatasourceConfig(name="jaffle_shop", type="duckdb", database=DB_PATH)
sa_engine = sa.create_engine(ds.resolve_env_vars().get_connection_string())
inspector = sa.inspect(sa_engine)
fk_graph = _build_fk_graph(inspector=inspector, table_names=inspector.get_table_names(), schema=None)
sa_engine.dispose()

print("FK graph:")
for table in sorted(fk_graph):
    print(f"  {table} -> {sorted(fk_graph[table])}")

items_model = next(m for m in models if m.name == "items")
orders_model = next(m for m in models if m.name == "orders")

print("\nitems joins (direct FKs only):")
for j in items_model.joins:
    src, tgt = j.join_pairs[0]
    print(f"  -> {j.target_model:<12} ON {src:<20} = {tgt}")

print("\norders joins (direct FKs only):")
for j in orders_model.joins:
    src, tgt = j.join_pairs[0]
    print(f"  -> {j.target_model:<12} ON {src:<20} = {tgt}")

print("\nMulti-hop: items reaches customers via orders.joins")

FK graph:
  items -> ['orders', 'products']
  orders -> ['customers', 'stores']
  supplies -> ['products']
  tweets -> ['customers']

items joins (direct FKs only):
  -> orders       ON order_id             = id
  -> products     ON sku                  = sku

orders joins (direct FKs only):
  -> customers    ON customer_id          = id
  -> stores       ON store_id             = id

Multi-hop: items reaches customers via orders.joins


## Diamond Joins

A **diamond join** occurs when the same table is reachable via multiple FK paths. For example:

```
orders -> customers -> regions
orders -> warehouses -> regions
```

SLayer treats each path as a separate copy of the target table, using **path-based aliases** to disambiguate:
- `customers.regions.name` -> table alias `customers__regions`  
- `warehouses.regions.name` -> table alias `warehouses__regions`

The Jaffle Shop schema doesn't have a natural diamond, so let's construct another schema for illustration:

In [7]:
from joins_utils import setup_diamond_example

diamond_engine, diamond_storage, diamond_models, diamond_db_path, diamond_work_dir = setup_diamond_example()

diamond_orders = next(m for m in diamond_models if m.name == "orders")
diamond_customers = next(m for m in diamond_models if m.name == "customers")
diamond_warehouses = next(m for m in diamond_models if m.name == "warehouses")

print("orders joins (direct FKs only):")
for j in diamond_orders.joins:
    src, tgt = j.join_pairs[0]
    print(f"  -> {j.target_model:<12} ON {src:<25} = {tgt}")

print("\ncustomers joins:")
for j in diamond_customers.joins:
    src, tgt = j.join_pairs[0]
    print(f"  -> {j.target_model:<12} ON {src:<25} = {tgt}")

print("\nwarehouses joins:")
for j in diamond_warehouses.joins:
    src, tgt = j.join_pairs[0]
    print(f"  -> {j.target_model:<12} ON {src:<25} = {tgt}")

print("\nDiamond: orders reaches regions via customers.joins AND warehouses.joins")
print("Each path gets a unique alias: customers__regions vs warehouses__regions")

orders joins (direct FKs only):
  -> customers    ON customer_id               = id
  -> warehouses   ON warehouse_id              = id

customers joins:
  -> regions      ON region_id                 = id

warehouses joins:
  -> regions      ON region_id                 = id

Diamond: orders reaches regions via customers.joins AND warehouses.joins
Each path gets a unique alias: customers__regions vs warehouses__regions


In [8]:
# Query both paths simultaneously
result = diamond_engine.execute_sync(
    query={
        "source_model": "orders",
        "measures": ["count(*)", "sum(amount)"],
        "dimensions": [
            "customers.regions.name",
            "warehouses.regions.name",
        ],
    }
)

print(f"{'Customer Region':<18} {'Warehouse Region':<18} {'Orders':>7} {'Amount':>10}")
print("-" * 56)
for row in result.data:
    cr = row["orders.customers.regions.name"]
    wr = row["orders.warehouses.regions.name"]
    print(f"{cr:<18} {wr:<18} {row['orders._count']:>7} ${row['orders.amount_sum']:>9,.2f}")

Customer Region    Warehouse Region    Orders     Amount
--------------------------------------------------------
East               West                     2 $   600.00
West               East                     3 $   750.00
West               West                     1 $   175.00
East               Central                  2 $   375.00
Central            Central                  1 $   150.00
Central            West                     1 $   225.00


### Recombining Diamond Joins with Filters

By default, each path to the same table produces independent copies. If you want to enforce that the two paths resolve to the same row (re-creating a true diamond), add a query filter equating the two paths — filters take the same dotted join-path syntax as dimensions:

If you want to make that constraint permanent, add the same expression to the `filters` of the model in question (`orders` in the example below) instead of to each query.

In [9]:
# Only keep orders where customer and warehouse are in the same region.
result = diamond_engine.execute_sync(
    query={
        "source_model": "orders",
        "measures": ["count(*)", "sum(amount)"],
        "dimensions": ["customers.regions.name"],
        "filters": ["customers.regions.name = warehouses.regions.name"],
    }
)

print("Orders where customer and warehouse are in the same region:")
for row in result.data:
    print(f"  {row['orders.customers.regions.name']}: {row['orders._count']} orders, ${row['orders.amount_sum']:,.2f}")

Orders where customer and warehouse are in the same region:
  West: 1 orders, $175.00
  Central: 1 orders, $150.00


## Dynamic Joins (ModelExtension)

Joins can be added at query time via `ModelExtension`, without modifying the stored model. This is useful for:
- Ad-hoc joins to lookup tables
- Joining to models created dynamically from queries
- Adding context-specific enrichment

More on this in the post (and companion notebook) on [multistage queries](../06_multistage_queries/multistage_queries.md).

## Summary

SLayer's join system provides:

| Feature | Description |
|---------|-------------|
| **LEFT JOINs** | The only join type, used for data enrichment |
| **Dot syntax** | `customers.name`, `orders.customers.regions.name` in queries |
| **Generated `__` aliases** | Internal table aliases in emitted SQL (`... AS orders__customers`); you always write dotted paths |
| **Auto-ingestion** | FK constraints become `ModelJoin` objects automatically |
| **Diamond joins** | Same table via multiple paths gets separate path-based aliases |
| **Dynamic joins** | `ModelExtension` adds joins at query time without modifying models |

See the [Models docs](../../concepts/models.md#joins) and [Ingestion docs](../../concepts/ingestion.md) for the full reference.